# Wav2Lip Lip-Sync on Colab **T4**

Makes a face (video or image) speak an audio clip. Perfect for making a
Wan-animated character talk with your Floyd / Cuffem / Player audio.

### This runs on the Colab T4 GPU only — NOT HuggingFace ZeroGPU.
It clones the open-source Wav2Lip code and runs `inference.py` on the T4
that Colab assigns to *your* runtime. It never calls the hosted HF Space,
so ZeroGPU (HF's shared, quota'd A100 slices) is never involved.

**First:** Runtime -> Change runtime type -> **T4 GPU** -> Save. Then run the
cells top to bottom. Step 0 proves you actually got a T4.

> **Small face?** If the mouth won't move, the face is too small for the
> detector. Use Steps 3.5 (upscale) and/or the CROP / BOX knobs in Step 4 to
> make the model focus on the face.


## Step 0 - Prove the GPU is a T4 (not ZeroGPU, not CPU)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then rerun.'
name = torch.cuda.get_device_name(0)
print('Torch is using:', name)
print('VERIFIED: running on', name, '- this is Colab hardware, not HF ZeroGPU.')


## Step 1 - Get Wav2Lip (maintained fork) and install deps
Uses `justinjohn0306/Wav2Lip`, which fixes the old-dependency problems and
hosts the model checkpoints.


In [ ]:
import os
if not os.path.isdir('/content/Wav2Lip'):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip /content/Wav2Lip
%cd /content/Wav2Lip
!pip install -q -r requirements.txt
!pip install -q batch-face gdown
print('deps installed')


## Step 2 - Download the model checkpoints into the runtime


In [ ]:
%cd /content/Wav2Lip
import os
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('face_detection/detection/sfd', exist_ok=True)
B='https://github.com/justinjohn0306/Wav2Lip/releases/download/models/'
!wget -q -c $B'wav2lip.pth'     -O checkpoints/wav2lip.pth
!wget -q -c $B'wav2lip_gan.pth' -O checkpoints/wav2lip_gan.pth
!wget -q -c $B's3fd.pth'        -O face_detection/detection/sfd/s3fd.pth
!wget -q -c $B'mobilenet.pth'   -O checkpoints/mobilenet.pth
!wget -q -c 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth' -O checkpoints/GFPGANv1.4.pth
print('checkpoints ready:', os.listdir('checkpoints'))


## Step 3 - Upload your inputs
**FACE** = a video or image of the character (e.g. a Wan clip from
`D:\\MatrixVideos`, or the close-up crop). **AUDIO** = the voice line
(wav/mp3, e.g. floyddeath.wav).

TIP: a **close-up** where the face fills much of the frame works far better
than a wide/full-body shot.


In [ ]:
from google.colab import files
print('Upload the FACE (mp4 / png / jpg):')
FACE = '/content/Wav2Lip/' + list(files.upload().keys())[0]
print('Upload the AUDIO (wav / mp3):')
AUDIO = '/content/Wav2Lip/' + list(files.upload().keys())[0]
print('FACE :', FACE)
print('AUDIO:', AUDIO)


## Step 3.5 (optional) - Upscale the input so a small face is detectable
If the mouth won't move, the face has too few pixels. Enlarging the whole
frame gives the detector more to work with. Works for both video and image.
Set `UPSCALE = 1` to skip. 2-3 is usually enough.


In [ ]:
UPSCALE = 1   # 1 = off. Try 2 or 3 if the face is small / detection fails.
if UPSCALE > 1:
    import os
    ext = os.path.splitext(FACE)[1].lower()
    up  = '/content/Wav2Lip/upscaled' + ext
    os.system(f'ffmpeg -y -loglevel error -i "{FACE}" -vf "scale=iw*{UPSCALE}:ih*{UPSCALE}:flags=lanczos" "{up}"')
    FACE = up
    print('upscaled', UPSCALE, 'x ->', FACE)
else:
    print('upscale skipped (UPSCALE=1)')


## Step 4 - Run Wav2Lip on the T4 (with focus knobs)
Three ways to make the model focus on the face:
- **CROP** `[top, bottom, left, right]` (pixels): zoom into a sub-region of
  the frame *before* detection. Best when the character sits in one part of
  a big frame. `None` = whole frame.
- **BOX** `[top, bottom, left, right]` (pixels): force an exact face box and
  skip detection entirely. Use only when detection fails completely.
- **PADS** `[top, bottom, left, right]`: grow the detected box; increase
  bottom if the chin/mouth is cut off.

`wav2lip_gan.pth` gives the best mouth quality; `--nosmooth` helps single faces.


In [ ]:
CROP = None            # e.g. [285, 620, 495, 675] to zoom into the character
BOX  = None            # e.g. [300, 380, 540, 640] to FORCE the face box
PADS = [0, 15, 0, 0]   # top bottom left right

cmd = ['python','inference.py',
       '--checkpoint_path','checkpoints/wav2lip_gan.pth',
       '--face', FACE, '--audio', AUDIO,
       '--outfile','/content/result.mp4',
       '--nosmooth','--resize_factor','1',
       '--pads', *map(str, PADS)]
if CROP: cmd += ['--crop', *map(str, CROP)]
if BOX:  cmd += ['--box',  *map(str, BOX)]
print('running:', ' '.join(cmd))
import subprocess; subprocess.run(cmd, check=True)
print('done -> /content/result.mp4')


## Step 5 - Preview and download the result


In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open('/content/result.mp4','rb').read()).decode()
HTML(f'<video width=480 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')


In [ ]:
from google.colab import files
files.download('/content/result.mp4')  # saves to your Downloads; move it to D:\\MatrixVideos


---
### Finding CROP / BOX pixel values
Open your FACE image, read off the pixel rectangle around the head. CROP
zooms the whole frame to that rectangle (keeps context); BOX tells Wav2Lip
exactly where the face is (use if detection still fails after CROP+upscale).

### Sharper faces with GFPGAN
If the face looks soft, re-run Step 4 adding `--out_height 720` (GFPGAN was
downloaded in Step 2). Higher = slower on the T4.
